# Reproduce the paired benchmark

Run on a Colab T4. Download the adapter from the GitHub repository and upload it when prompted. This notebook regenerates the 54 image pairs and detector predictions and checks the same 72 real reference images. Download the resulting files from the Colab file browser under `/content/pmldl_coco/portfolio/`.

In [ ]:
%pip -q install diffusers==0.35.2 transformers==4.56.2 peft==0.17.1 accelerate==1.10.1
import gc
import io
import json
import os
import random
import re
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from textwrap import fill
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from huggingface_hub import model_info
from IPython.display import Markdown, display
from PIL import Image, ImageOps
from torch.nn import functional as F
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Select a T4 GPU in Colab before running."
DEVICE = torch.device("cuda")
DTYPE = torch.float16
SEED = 2026
torch.manual_seed(SEED)
ROOT = Path("/content/pmldl_coco")
ROOT.mkdir(exist_ok=True)
COCO = "https://s3.amazonaws.com/images.cocodataset.org"
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RESOLUTION = 384
TRAIN_PER_CLASS, VAL_PER_CLASS = 100, 12
STEPS, ACCUMULATION, LR = 240, 2, 1e-4
print(f"PyTorch {torch.__version__} | {torch.cuda.get_device_name()} | {RESOLUTION} × {RESOLUTION}")

In [ ]:
FOODS = {
    "pizza": r"pizzas?", "sandwich": r"sandwich(?:es)?", "cake": r"cakes?",
    "donut": r"(?:doughnut|donut)s?", "banana": r"bananas?", "broccoli": r"broccoli",
}

archive = ROOT / "annotations_trainval2017.zip"
if not archive.exists():
    with requests.get(f"{COCO}/annotations/{archive.name}", stream=True, timeout=(15, 90)) as response:
        response.raise_for_status()
        temp = archive.with_suffix(".partial")
        with temp.open("wb") as output:
            for block in response.iter_content(1 << 20):
                output.write(block)
    temp.replace(archive)

def candidates_for(split, seed):
    with ZipFile(archive) as source:
        with source.open(f"annotations/instances_{split}.json") as file:
            instances = json.load(file)
        with source.open(f"annotations/captions_{split}.json") as file:
            captions = json.load(file)
    category_ids = {c["id"]: c["name"] for c in instances["categories"] if c["name"] in FOODS}
    labels = defaultdict(set)
    for annotation in instances["annotations"]:
        category = category_ids.get(annotation["category_id"])
        if category and not annotation["iscrowd"]:
            labels[annotation["image_id"]].add(category)
    del instances
    names = {i["id"]: i["file_name"] for i in captions["images"]
             if min(i["width"], i["height"]) >= 200}
    groups = {food: defaultdict(list) for food in FOODS}
    for row in captions["annotations"]:
        image_id = row["image_id"]
        if image_id not in names:
            continue
        caption = " ".join(row["caption"].split())
        for food in sorted(labels.get(image_id, ())):
            if re.search(rf"\b{FOODS[food]}\b", caption, flags=re.I):
                groups[food][image_id].append(caption)
    rng = random.Random(seed)
    result = {}
    for food, images in groups.items():
        ids = sorted(images)
        rng.shuffle(ids)
        result[food] = [(image_id, names[image_id], sorted(set(images[image_id])))
                        for image_id in ids]
    return result

def fetch_image(task):
    split, filename = task
    path = ROOT / split / filename
    if not path.exists():
        for attempt in range(3):
            try:
                response = requests.get(f"{COCO}/{split}/{filename}", timeout=(8, 18))
                response.raise_for_status()
                with Image.open(io.BytesIO(response.content)) as image:
                    image.verify()
                path.write_bytes(response.content)
                break
            except (requests.RequestException, OSError):
                if attempt == 2:
                    return None
                time.sleep(0.25 * (attempt + 1))
    return path

def load_split(split, per_class, seed):
    candidates = candidates_for(split, seed)
    (ROOT / split).mkdir(exist_ok=True)
    used = set()
    rows = []
    with ThreadPoolExecutor(max_workers=20) as pool:
        for food, images in candidates.items():
            count = 0
            available = [r for r in images if r[0] not in used]
            for offset in range(0, len(available), per_class):
                batch = available[offset:offset + per_class]
                paths = pool.map(fetch_image, [(split, row[1]) for row in batch])
                for row, path in zip(batch, paths):
                    if path is not None and count < per_class:
                        image_id, _, captions = row
                        rows.append((image_id, food, path, captions))
                        used.add(image_id)
                        count += 1
                if count == per_class:
                    break
            if count < per_class:
                raise RuntimeError(f"{split}: downloaded only {count}/{per_class} {food} images")
            print(f"{split:9s} | {food:8s} | {count} usable images", flush=True)
    random.Random(seed + 100).shuffle(rows)
    return rows

def crop_photo(path):
    with Image.open(path) as opened:
        rgb = ImageOps.exif_transpose(opened).convert("RGB")
        return ImageOps.fit(rgb, (RESOLUTION, RESOLUTION), Image.Resampling.BICUBIC)


val = load_split("val2017", VAL_PER_CLASS, SEED + 1)

In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler, DDIMScheduler, UNet2DConditionModel
from peft import LoraConfig, get_peft_model_state_dict
from safetensors.torch import save_file
from transformers import CLIPTextModel, CLIPTokenizer

from hashlib import sha256
from google.colab import files
from peft import set_peft_model_state_dict
from safetensors.torch import load_file

adapter_file = Path("/content/coco-food-lora.safetensors")
if not adapter_file.exists():
    uploaded = files.upload()
    if adapter_file.name not in uploaded:
        raise ValueError(f"Upload {adapter_file.name}.")
    adapter_file.write_bytes(uploaded[adapter_file.name])
assert sha256(adapter_file.read_bytes()).hexdigest() == "8bad61aaa803fafb20446c6080b686f97a4a2500e447d70a56e24aa995aaa008"

revision = '451f4fe16113bff5a5d2269ed5ad43b0592e9a14'
print("Stable Diffusion revision:", revision, flush=True)
options = dict(revision=revision, torch_dtype=DTYPE, variant="fp16", use_safetensors=True)
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer", revision=revision)
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", **options)
text_encoder.eval().requires_grad_(False).to(DEVICE)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", **options)
vae.eval().requires_grad_(False).to(DEVICE)
schedule = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler", revision=revision)
assert schedule.config.prediction_type == "epsilon"

@torch.no_grad()
def encode_text(prompts):
    token_ids = tokenizer(prompts, padding="max_length", max_length=tokenizer.model_max_length,
                          truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    return text_encoder(token_ids).last_hidden_state

unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", **options)
unet.requires_grad_(False).to(DEVICE)
target = r".*\.attn2\.(to_q|to_k|to_v|to_out\.0)"
unet.add_adapter(LoraConfig(r=8, lora_alpha=8, init_lora_weights="gaussian", target_modules=target))

trainable = [p for p in unet.parameters() if p.requires_grad]
for p in trainable:
    p.data = p.data.float()
state = load_file(str(adapter_file))
assert state.keys() == get_peft_model_state_dict(unet).keys()
loaded = set_peft_model_state_dict(unet, state)
assert not loaded.unexpected_keys and not any("lora_" in key for key in loaded.missing_keys)
assert all(torch.equal(get_peft_model_state_dict(unet)[key].cpu(), value) for key, value in state.items())
unet.eval()


In [ ]:
@torch.no_grad()
def sample(batch, collect_trace=False):
    negative = encode_text([""] * len(batch))
    positive = encode_text([prompt for _, prompt, _ in batch])
    embeds = torch.cat([negative, positive])
    scheduler = DDIMScheduler.from_config(schedule.config)
    scheduler.set_timesteps(24, device=DEVICE)
    latent = torch.stack([
        torch.randn((4, RESOLUTION // 8, RESOLUTION // 8), device=DEVICE, dtype=DTYPE,
                    generator=torch.Generator(device=DEVICE).manual_seed(seed))
        for _, _, seed in batch
    ]) * scheduler.init_noise_sigma
    trace = [latent[:1].clone()] if collect_trace else []
    for i, timestep in enumerate(scheduler.timesteps, 1):
        latent_input = scheduler.scale_model_input(torch.cat([latent, latent]), timestep)
        with torch.autocast("cuda", dtype=DTYPE):
            uncond, cond = unet(latent_input, timestep, encoder_hidden_states=embeds).sample.chunk(2)
        guided = uncond + 7.5 * (cond - uncond)
        latent = scheduler.step(guided, timestep, latent, eta=0.0).prev_sample
        if collect_trace and i in (6, 12, 18, 24):
            trace.append(latent[:1].clone())

    def decode(z):
        x = vae.decode(z / vae.config.scaling_factor).sample
        assert torch.isfinite(x).all()
        x = x.float().mul(.5).add(.5).clamp(0, 1)
        return [Image.fromarray(a) for a in x.mul(255).round().byte().permute(0, 2, 3, 1).cpu().numpy()]

    return decode(latent), decode(torch.cat(trace)) if trace else []

def generate_all(tasks, with_trace=False):
    images = []
    trace = []
    for i in tqdm(range(0, len(tasks), 3), desc="DDIM groups"):
        output, frames = sample(tasks[i:i+3], collect_trace=with_trace and i == 0)
        images.extend(output)
        trace.extend(frames)
    return images, trace



In [ ]:
import csv
import hashlib
import zipfile
from transformers import DetrForObjectDetection, DetrImageProcessor

bench = []
for food in ['pizza', 'sandwich', 'cake', 'donut', 'banana', 'broccoli']:
    for scene, template in enumerate(['A close-up photo of {food} served on a white plate.', 'A restaurant table with {food} on a blue plate.']):
        bench.append(dict(id=f'identity-{food}-{scene}', task='identity', food=food, count='', side='', prompt=template.format(food=food)))
for food, plural in [('donut', 'donuts'), ('banana', 'bananas'), ('sandwich', 'sandwiches')]:
    for count, word in enumerate(['One', 'Two', 'Three'], 1):
        bench.append(dict(id=f'count-{food}-{count}', task='count', food=food, count=count, side='', prompt=f'{word} whole {food if count == 1 else plural} on a white plate, photographed from directly above.'))
for food in ['pizza', 'sandwich', 'donut']:
    for side in ['left', 'right']:
        other = 'right' if side == 'left' else 'left'
        bench.append(dict(id=f'spatial-{food}-{side}', task='spatial', food=food, count='', side=side, prompt=f'A {food} on the {side} of a white coffee cup on the {other}, overhead food photography.'))
orders = [(p, seed) for p in bench for seed in (732, 984)]
assert len(bench) == 27 and len(orders) == 54

unet.to(DEVICE).eval()
vae.to(DEVICE).eval()
text_encoder.to(DEVICE).eval()
bench_jobs = [(row['food'], row['prompt'], seed) for row, seed in orders]
unet.disable_adapters()
bench_base, _ = generate_all(bench_jobs)
unet.enable_adapters()
bench_tuned, _ = generate_all(bench_jobs)
print('Generated 54 pairs.', flush=True)

unet.to('cpu'); vae.to('cpu'); text_encoder.to('cpu')
gc.collect(); torch.cuda.empty_cache()
DETECTOR = 'facebook/detr-resnet-50'
detr_revision = '1d5f47bd3bdd2c4bbfa585418ffe6da5028b4c0b'
image_processor = DetrImageProcessor.from_pretrained(DETECTOR, revision=detr_revision)
detector = DetrForObjectDetection.from_pretrained(DETECTOR, revision=detr_revision).to(DEVICE).eval()
print('DETR revision:', detr_revision, flush=True)

@torch.no_grad()
def detect(images):
    records = []
    for start in range(0, len(images), 8):
        group = images[start:start + 8]
        inputs = image_processor(images=group, return_tensors='pt').to(DEVICE)
        outputs = detector(**inputs)
        sizes = torch.tensor([(p.height, p.width) for p in group], device=DEVICE)
        predictions = image_processor.post_process_object_detection(outputs, threshold=0.30, target_sizes=sizes)
        for result in predictions:
            records.append([dict(label=detector.config.id2label[int(label)], confidence=round(float(score), 6),
                                 box=[round(x, 3) for x in box.tolist()])
                            for label, score, box in zip(result['labels'].cpu(),result['scores'].cpu(),result['boxes'].cpu())])
    return records

base_detections = detect(bench_base)
tuned_detections = detect(bench_tuned)
reference_detections = detect([crop_photo(row[2]) for row in val])
report_dir = ROOT / 'portfolio'
report_dir.mkdir(exist_ok=True)
with (report_dir / 'prompts.csv').open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=list(bench[0])); writer.writeheader(); writer.writerows(bench)
(report_dir / 'detections.json').write_text(json.dumps(dict(
    detector=DETECTOR, detector_revision=detr_revision, seeds=[732,984],
    adapter_sha256=hashlib.sha256(adapter_file.read_bytes()).hexdigest(),
    samples=[dict(id=p['id'], seed=seed, pretrained=a, tuned=b)
             for (p,seed),a,b in zip(orders,base_detections,tuned_detections)],
    reference=[dict(coco_id=row[0],food=row[1],detections=d) for row,d in zip(val,reference_detections)]),indent=2))
with zipfile.ZipFile(report_dir / 'coco-benchmark.zip','w') as output:
    for title, photos in [('pretrained',bench_base),('tuned',bench_tuned)]:
        for (row,seed), photo in zip(orders,photos):
            with io.BytesIO() as buffer:
                photo.save(buffer,format='JPEG',quality=85)
                output.writestr(f'{row["id"]}-{seed}-{title}.jpg',buffer.getvalue())
print('Benchmark artifacts saved.', flush=True)